# Task 3: Developing multi-agent systems

## Goal

Build and test a Google ADK coordinator that delegates U.S. weather
work to a custom-tool specialist and current-information research to
a Google Search specialist, with saved events that prove each handoff.

## Checklist

- [x] Start from the completed Task 2 notebook and preserve its weather tools.
- [x] Create a coordinating root agent.
- [x] Create a weather agent with Google Maps and NWS function tools.
- [x] Create a search agent with ADK's built-in Google Search tool.
- [x] Register both specialists as root-agent sub-agents.
- [x] Test weather-only, current-search, combined, unrelated, and failure cases.
- [x] Save transfer, event-author, tool-call, grounding, and final-response evidence.
- [x] Use fresh ADK sessions and grade every acceptance criterion.

- **Project:** `qwiklabs-gcp-02-66b2cfb8579b`
- **Region:** `us-central1`
- **Model:** `gemini-2.5-flash`


## 1. Task 2 foundation

This notebook is programmatically copied forward from
`02_callbacks.ipynb`. It reuses the executed Task 2 dependency,
authentication, Google Maps Geocoding, and National Weather Service
implementation. Task 3 adds the new search specialist, root
coordinator, delegation trace, and grading scenarios.


In [1]:
import importlib.util
import subprocess
import sys


required_modules = ("google.adk", "requests")
missing_modules = [
    module for module in required_modules if importlib.util.find_spec(module) is None
]
if missing_modules:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "google-adk>=1.18,<2.0",
            "requests>=2.32,<3",
        ],
        check=True,
    )
    print(f"Installed missing modules: {missing_modules}")
else:
    print("Required Python modules are already installed.")


Required Python modules are already installed.


In [2]:
from __future__ import annotations

import importlib.metadata
import json
import os
import subprocess
import uuid
from typing import Any

import google.auth
import requests


EXPECTED_PROJECT = "qwiklabs-gcp-02-66b2cfb8579b"
LOCATION = "us-central1"
MODEL = "gemini-2.5-flash"


def run_gcloud(arguments: list[str]) -> subprocess.CompletedProcess[str]:
    """Run a bounded gcloud command without printing credentials."""
    return subprocess.run(
        ["gcloud", *arguments],
        check=False,
        capture_output=True,
        text=True,
        timeout=30,
    )


project_result = run_gcloud(["config", "get-value", "project"])
detected_project = project_result.stdout.strip()
_, adc_project = google.auth.default()
observed_projects = {value for value in (detected_project, adc_project) if value}

print(
    json.dumps(
        {
            "expected_project": EXPECTED_PROJECT,
            "gcloud_project": detected_project,
            "adc_project": adc_project,
            "location": LOCATION,
            "model": MODEL,
            "google_adk_version": importlib.metadata.version("google-adk"),
        },
        indent=2,
    )
)

if observed_projects != {EXPECTED_PROJECT}:
    raise RuntimeError(
        f"Project mismatch: expected {EXPECTED_PROJECT}, observed {observed_projects}"
    )

os.environ["GOOGLE_CLOUD_PROJECT"] = EXPECTED_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"


{
  "expected_project": "qwiklabs-gcp-02-66b2cfb8579b",
  "gcloud_project": "qwiklabs-gcp-02-66b2cfb8579b",
  "adc_project": "qwiklabs-gcp-02-66b2cfb8579b",
  "location": "us-central1",
  "model": "gemini-2.5-flash",
  "google_adk_version": "1.39.0"
}


In [3]:
MAPS_KEY_DISPLAY_NAME = "task1-weather-geocoding-v2"


def load_maps_api_key() -> str:
    """Load the Maps key from the environment or Google API Keys service."""
    environment_key = os.getenv("GOOGLE_MAPS_API_KEY", "").strip()
    if environment_key:
        return environment_key

    list_result = run_gcloud(
        [
            "services",
            "api-keys",
            "list",
            f"--filter=displayName={MAPS_KEY_DISPLAY_NAME}",
            "--format=value(name)",
        ]
    )
    key_names = [line.strip() for line in list_result.stdout.splitlines() if line.strip()]
    if list_result.returncode or not key_names:
        raise RuntimeError(
            "A restricted Google Maps key named "
            f"{MAPS_KEY_DISPLAY_NAME!r} is required."
        )

    key_result = run_gcloud(
        [
            "services",
            "api-keys",
            "get-key-string",
            key_names[0],
            "--format=value(keyString)",
        ]
    )
    key_string = key_result.stdout.strip()
    if key_result.returncode or not key_string:
        raise RuntimeError("The Maps key exists but its key string could not be loaded.")
    return key_string


GOOGLE_MAPS_API_KEY = load_maps_api_key()
print({"maps_credential_loaded": bool(GOOGLE_MAPS_API_KEY)})


{'maps_credential_loaded': True}


## 2. Reused weather tools

The weather specialist keeps the typed, documented Task 2 tools.
External calls have explicit timeouts and return compact sanitized
objects without logging credentials or raw authenticated URLs.


In [4]:
MAPS_GEOCODING_URL = "https://maps.googleapis.com/maps/api/geocode/json"
NWS_API_ROOT = "https://api.weather.gov"
REQUEST_TIMEOUT_SECONDS = 20
NWS_HEADERS = {
    "Accept": "application/geo+json",
    "User-Agent": "task1-weather-agent/1.0 (Google Cloud skills workshop)",
}


class ExternalServiceError(RuntimeError):
    """Describe a safe external-service failure without including a secret URL."""


def request_json(
    url: str,
    *,
    service_name: str,
    params: dict[str, Any] | None = None,
    headers: dict[str, str] | None = None,
) -> dict[str, Any]:
    """Return JSON from an HTTP GET request or raise a sanitized error.

    Args:
        url: Service endpoint without user-facing logging.
        service_name: Safe name used in error messages.
        params: Optional query parameters.
        headers: Optional HTTP request headers.

    Returns:
        The decoded JSON object.

    Raises:
        ExternalServiceError: If the request or JSON decoding fails.
    """
    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
    except requests.RequestException as exc:
        raise ExternalServiceError(f"{service_name} request failed.") from exc

    if not response.ok:
        raise ExternalServiceError(
            f"{service_name} returned HTTP {response.status_code}."
        )
    try:
        payload = response.json()
    except ValueError as exc:
        raise ExternalServiceError(f"{service_name} returned invalid JSON.") from exc
    if not isinstance(payload, dict):
        raise ExternalServiceError(f"{service_name} returned an unexpected payload.")
    return payload


def geocode_place(place: str) -> dict[str, Any]:
    """Convert a U.S. place name to latitude and longitude with Google Maps.

    Args:
        place: A city, address, or named place in the United States.

    Returns:
        A compact dictionary with status, formatted address, coordinates, and
        place ID. Error results contain a safe message and no credential data.
    """
    normalized_place = place.strip()
    if not normalized_place:
        return {"status": "error", "message": "Place must not be empty."}

    try:
        payload = request_json(
            MAPS_GEOCODING_URL,
            service_name="Google Maps Geocoding API",
            params={
                "address": normalized_place,
                "components": "country:US",
                "key": GOOGLE_MAPS_API_KEY,
            },
        )
    except ExternalServiceError as exc:
        return {"status": "error", "message": str(exc)}

    api_status = payload.get("status")
    results = payload.get("results") or []
    if api_status != "OK" or not results:
        safe_status = str(api_status or "UNKNOWN")
        return {
            "status": "error",
            "message": f"Google Maps found no usable result ({safe_status}).",
        }

    first_result = results[0]
    result_types = set(first_result.get("types", []))
    if first_result.get("partial_match") or result_types <= {"country", "political"}:
        return {
            "status": "error",
            "message": "Google Maps returned only a partial or country-level match.",
        }
    country_codes = {
        component.get("short_name")
        for component in first_result.get("address_components", [])
        if "country" in component.get("types", [])
    }
    if country_codes != {"US"}:
        return {"status": "error", "message": "The result is outside the United States."}

    location = first_result["geometry"]["location"]
    return {
        "status": "success",
        "query": normalized_place,
        "formatted_address": first_result.get("formatted_address"),
        "latitude": round(float(location["lat"]), 6),
        "longitude": round(float(location["lng"]), 6),
        "place_id": first_result.get("place_id"),
    }


In [5]:
def celsius_to_fahrenheit(value: float | None) -> float | None:
    """Convert Celsius to Fahrenheit when a value is present."""
    return None if value is None else round((value * 9 / 5) + 32, 1)


def meters_per_second_to_mph(value: float | None) -> float | None:
    """Convert meters per second to miles per hour when a value is present."""
    return None if value is None else round(value * 2.23694, 1)


def measurement_value(properties: dict[str, Any], name: str) -> float | None:
    """Read a numeric NWS observation measurement when available."""
    measurement = properties.get(name) or {}
    value = measurement.get("value")
    return float(value) if isinstance(value, (int, float)) else None


def get_weather(latitude: float, longitude: float) -> dict[str, Any]:
    """Get current NWS observations, forecast, and alerts for coordinates.

    Args:
        latitude: Latitude in decimal degrees from -90 through 90.
        longitude: Longitude in decimal degrees from -180 through 180.

    Returns:
        Current observation data, the nearest forecast period, and up to five
        active NWS alerts. Errors contain a safe, concise message.
    """
    if not -90 <= latitude <= 90:
        return {"status": "error", "message": "Latitude must be between -90 and 90."}
    if not -180 <= longitude <= 180:
        return {
            "status": "error",
            "message": "Longitude must be between -180 and 180.",
        }

    point = f"{latitude:.4f},{longitude:.4f}"
    try:
        point_payload = request_json(
            f"{NWS_API_ROOT}/points/{point}",
            service_name="NWS points service",
            headers=NWS_HEADERS,
        )
        point_properties = point_payload["properties"]

        forecast_payload = request_json(
            point_properties["forecast"],
            service_name="NWS forecast service",
            headers=NWS_HEADERS,
        )
        periods = forecast_payload.get("properties", {}).get("periods", [])
        if not periods:
            raise ExternalServiceError("NWS forecast service returned no periods.")

        observation: dict[str, Any] = {"available": False}
        station_collection = request_json(
            point_properties["observationStations"],
            service_name="NWS station service",
            headers=NWS_HEADERS,
        )
        station_urls = station_collection.get("observationStations", [])
        if station_urls:
            latest_payload = request_json(
                f"{station_urls[0]}/observations/latest",
                service_name="NWS observation service",
                headers=NWS_HEADERS,
            )
            latest = latest_payload.get("properties", {})
            observation = {
                "available": True,
                "station": station_urls[0].rsplit("/", 1)[-1],
                "timestamp": latest.get("timestamp"),
                "description": latest.get("textDescription"),
                "temperature_f": celsius_to_fahrenheit(
                    measurement_value(latest, "temperature")
                ),
                "humidity_percent": (
                    round(measurement_value(latest, "relativeHumidity"), 1)
                    if measurement_value(latest, "relativeHumidity") is not None
                    else None
                ),
                "wind_mph": meters_per_second_to_mph(
                    measurement_value(latest, "windSpeed")
                ),
            }

        alerts_payload = request_json(
            f"{NWS_API_ROOT}/alerts/active",
            service_name="NWS alerts service",
            params={"point": point},
            headers=NWS_HEADERS,
        )
        alerts = []
        for feature in alerts_payload.get("features", [])[:5]:
            properties = feature.get("properties", {})
            alerts.append(
                {
                    "event": properties.get("event"),
                    "severity": properties.get("severity"),
                    "urgency": properties.get("urgency"),
                    "headline": properties.get("headline"),
                    "instruction": properties.get("instruction"),
                }
            )
    except (ExternalServiceError, KeyError, TypeError, ValueError) as exc:
        message = str(exc) if isinstance(exc, ExternalServiceError) else "NWS response was incomplete."
        return {"status": "error", "message": message}

    current_period = periods[0]
    alert_summary = (
        "; ".join(alert.get("event") or "Weather alert" for alert in alerts)
        if alerts
        else "No active NWS alerts."
    )
    return {
        "status": "success",
        "coordinates": {"latitude": latitude, "longitude": longitude},
        "location": {
            "city": point_properties.get("relativeLocation", {})
            .get("properties", {})
            .get("city"),
            "state": point_properties.get("relativeLocation", {})
            .get("properties", {})
            .get("state"),
        },
        "observation": observation,
        "forecast": {
            "name": current_period.get("name"),
            "temperature": current_period.get("temperature"),
            "temperature_unit": current_period.get("temperatureUnit"),
            "wind": f"{current_period.get('windSpeed')} {current_period.get('windDirection')}",
            "short_forecast": current_period.get("shortForecast"),
            "detailed_forecast": current_period.get("detailedForecast"),
        },
        "active_alert_count": len(alerts),
        "alert_summary": alert_summary,
        "alerts": alerts,
    }


In [6]:
assert geocode_place("   ") == {
    "status": "error",
    "message": "Place must not be empty.",
}
assert get_weather(90.01, 0)["status"] == "error"
assert get_weather(0, -180.01)["status"] == "error"
assert geocode_place.__annotations__["place"] == "str"
assert get_weather.__annotations__["latitude"] == "float"
assert geocode_place.__doc__ and get_weather.__doc__
print("Deterministic validation checks: PASS")


Deterministic validation checks: PASS


## 3. Specialist agents and root coordinator

The search specialist contains only ADK's built-in
`google_search` tool, respecting its single-tool-per-agent
constraint. Clear descriptions give the root model reliable routing
signals. Both specialists are registered through `sub_agents`, so
ADK provides automatic transfer actions.


In [7]:
import asyncio

from google.adk.agents import Agent
from google.adk.agents.callback_context import CallbackContext
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types


TASK3_SOURCE_NOTEBOOK = "02_callbacks.ipynb"
AGENT_START_EVENTS: list[dict[str, str]] = []


def configured_tool_name(tool: Any) -> str:
    """Return the public name of a built-in, function, or wrapped tool."""
    wrapped_function = getattr(tool, "func", None)
    wrapped_name = getattr(wrapped_function, "__name__", None)
    if wrapped_name:
        return str(wrapped_name)
    direct_name = getattr(tool, "name", None)
    return str(direct_name or type(tool).__name__)


def make_agent_start_callback(agent_name: str):
    """Create a callback that records a bounded specialist-start event."""

    def record_agent_start(
        callback_context: CallbackContext,
    ) -> None:
        del callback_context
        AGENT_START_EVENTS.append(
            {"event": "agent_start", "agent": agent_name}
        )
        return None

    return record_agent_start


weather_agent = Agent(
    name="weather_agent",
    model=MODEL,
    description=(
        "Specialist for live U.S. weather observations, forecasts, "
        "and National Weather Service alerts for a named city and state."
    ),
    instruction="""
    You are the weather specialist in a multi-agent team.
    1. For every weather request, call geocode_place with the full U.S.
       location, then call get_weather with the returned coordinates.
    2. Report the resolved location, current observation when available,
       forecast, and active-alert status. Put urgent alerts first.
    3. Never use general web knowledge or invent weather.
    4. If the user also asks for current web research, finish the weather
       work, summarize it briefly, then transfer to root_agent so the
       coordinator can send the remaining work to google_search_agent.
    5. For weather-only requests, answer directly and concisely.
    """,
    tools=[geocode_place, get_weather],
    before_agent_callback=make_agent_start_callback("weather_agent"),
)

google_search_agent = Agent(
    name="google_search_agent",
    model=MODEL,
    description=(
        "Specialist for current facts, recent developments, official "
        "announcements, and web research using Google Search."
    ),
    instruction="""
    You are the current-information specialist in a multi-agent team.
    Use the built-in Google Search tool for every request. Prefer
    authoritative primary sources, state relevant dates, distinguish
    sourced facts from inference, and give a concise answer. For a
    combined request, incorporate the weather facts already present in
    the conversation and finish the answer after completing search.
    """,
    tools=[google_search],
    before_agent_callback=make_agent_start_callback(
        "google_search_agent"
    ),
    # A built-in Google Search tool cannot be mixed with ADK's
    # automatic transfer tools. The root can transfer into this
    # terminal specialist after any weather handoff is complete.
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
)

root_agent = Agent(
    name="root_agent",
    model=MODEL,
    description=(
        "Coordinator that routes requests to live U.S. weather and "
        "current-information Google Search specialists."
    ),
    instruction="""
    You coordinate exactly two specialists.
    - Delegate live U.S. weather, forecasts, observations, or alerts to
      weather_agent. Do not answer weather from memory.
    - Delegate current events, recent developments, official updates,
      or other web research to google_search_agent.
    - When a request needs both, delegate the weather portion first.
      After weather_agent returns control, delegate the remaining current
      research to google_search_agent, then provide a concise combined
      answer grounded in both specialists' results.
    - For requests outside both capabilities, refuse briefly and describe
      the supported weather and current-information capabilities.
    Do not call specialist tools yourself or fabricate specialist results.
    """,
    sub_agents=[weather_agent, google_search_agent],
    before_agent_callback=make_agent_start_callback("root_agent"),
)

APP_NAME = "task3_multi_agent_system"
USER_ID = "grader"
multi_agent_session_service = InMemorySessionService()
multi_agent_runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=multi_agent_session_service,
)

print(
    json.dumps(
        {
            "root_agent": root_agent.name,
            "sub_agents": [
                agent.name for agent in root_agent.sub_agents
            ],
            "weather_tools": [
                geocode_place.__name__,
                get_weather.__name__,
            ],
            "search_tools": [
                configured_tool_name(tool)
                for tool in google_search_agent.tools
            ],
            "model": MODEL,
        },
        indent=2,
    )
)


App "task3_multi_agent_system" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


{
  "root_agent": "root_agent",
  "sub_agents": [
    "weather_agent",
    "google_search_agent"
  ],
  "weather_tools": [
    "geocode_place",
    "get_weather"
  ],
  "search_tools": [
    "google_search"
  ],
  "model": "gemini-2.5-flash"
}


## 4. Observable event runner

ADK events are the grading log. Each record preserves the author,
hierarchy branch, transfer target, bounded function-call metadata,
Google Search grounding presence, and a short final-text preview.
Full provider responses and credentials are deliberately excluded.


In [8]:
SPECIALIST_NAMES = {"weather_agent", "google_search_agent"}


def bounded_text(value: str | None, limit: int = 320) -> str | None:
    """Normalize and truncate display text for a compact event log."""
    if not value:
        return None
    normalized = " ".join(value.split())
    return normalized[:limit] + ("..." if len(normalized) > limit else "")


def event_record(event: Any) -> dict[str, Any]:
    """Convert an ADK event into a compact, credential-free record."""
    calls = [
        {
            "name": call.name,
            "arguments": dict(call.args or {}),
        }
        for call in event.get_function_calls()
    ]
    responses = []
    for response in event.get_function_responses():
        payload = response.response
        responses.append(
            {
                "name": response.name,
                "status": (
                    payload.get("status")
                    if isinstance(payload, dict)
                    else None
                ),
            }
        )

    text = ""
    if event.content and event.content.parts:
        text = "".join(
            part.text or "" for part in event.content.parts if part.text
        )

    transfer_target = None
    if event.actions:
        transfer_target = event.actions.transfer_to_agent

    grounding = bool(getattr(event, "grounding_metadata", None))
    return {
        "author": event.author,
        "branch": event.branch,
        "transfer_to_agent": transfer_target,
        "function_calls": calls,
        "function_responses": responses,
        "google_search_grounding": grounding,
        "is_final_response": event.is_final_response(),
        "text_preview": bounded_text(text),
    }


async def run_multi_agent_case(
    prompt: str,
    *,
    label: str,
) -> dict[str, Any]:
    """Run one fresh-session root-agent turn and return its event trace."""
    session_id = f"task3-{label}-{uuid.uuid4().hex[:12]}"
    await multi_agent_session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )
    AGENT_START_EVENTS.clear()
    message = types.Content(
        role="user",
        parts=[types.Part.from_text(text=prompt)],
    )

    records: list[dict[str, Any]] = []
    final_answer = ""
    async with asyncio.timeout(120):
        async for event in multi_agent_runner.run_async(
            user_id=USER_ID,
            session_id=session_id,
            new_message=message,
        ):
            record = event_record(event)
            records.append(record)
            if record["is_final_response"] and record["text_preview"]:
                final_answer = " ".join(
                    part.text or ""
                    for part in event.content.parts
                    if part.text
                ).strip()

    event_authors = list(
        dict.fromkeys(
            record["author"]
            for record in records
            if record["author"]
        )
    )
    transfer_targets = [
        record["transfer_to_agent"]
        for record in records
        if record["transfer_to_agent"]
    ]
    tool_calls = [
        {
            "author": record["author"],
            **call,
        }
        for record in records
        for call in record["function_calls"]
        if call["name"] != "transfer_to_agent"
    ]
    started_agents = list(
        dict.fromkeys(item["agent"] for item in AGENT_START_EVENTS)
    )

    return {
        "label": label,
        "prompt": prompt,
        "session_id": session_id,
        "started_agents": started_agents,
        "event_authors": event_authors,
        "transfer_targets": transfer_targets,
        "tool_calls": tool_calls,
        "google_search_grounded": any(
            record["google_search_grounding"] for record in records
        ),
        "final_answer": final_answer,
        "events": records,
    }


## 5. Deterministic architecture checks

These checks prove the three-agent hierarchy and each specialist's
tool boundary without calling Gemini or an external service.


In [9]:
hierarchy_evidence = {
    "root_name": root_agent.name,
    "sub_agent_names": [agent.name for agent in root_agent.sub_agents],
    "weather_tool_names": [
        geocode_place.__name__,
        get_weather.__name__,
    ],
    "weather_agent_tool_count": len(weather_agent.tools),
    "search_tool_names": [
        configured_tool_name(tool)
        for tool in google_search_agent.tools
    ],
    "search_uses_builtin_google_search": (
        len(google_search_agent.tools) == 1
        and google_search_agent.tools[0] is google_search
    ),
    "search_is_terminal_specialist": bool(
        google_search_agent.disallow_transfer_to_parent
        and google_search_agent.disallow_transfer_to_peers
    ),
}

assert hierarchy_evidence["root_name"] == "root_agent"
assert hierarchy_evidence["sub_agent_names"] == [
    "weather_agent",
    "google_search_agent",
]
assert set(hierarchy_evidence["weather_tool_names"]) == {
    "geocode_place",
    "get_weather",
}
assert hierarchy_evidence["weather_agent_tool_count"] == 2
assert hierarchy_evidence["search_uses_builtin_google_search"] is True
assert hierarchy_evidence["search_is_terminal_specialist"] is True
assert weather_agent.parent_agent is root_agent
assert google_search_agent.parent_agent is root_agent
print(json.dumps(hierarchy_evidence, indent=2))


{
  "root_name": "root_agent",
  "sub_agent_names": [
    "weather_agent",
    "google_search_agent"
  ],
  "weather_tool_names": [
    "geocode_place",
    "get_weather"
  ],
  "weather_agent_tool_count": 2,
  "search_tool_names": [
    "google_search"
  ],
  "search_uses_builtin_google_search": true,
  "search_is_terminal_specialist": true
}


## 6. Live delegation scenarios

Every scenario starts with the root agent in a fresh ADK session.
Assertions require the appropriate specialist to appear as an event
author, not merely in configuration. The combined case must exercise
both specialists. The unrelated case must stay with the coordinator.


In [10]:
LIVE_CASES = [
    {
        "label": "weather_only",
        "prompt": (
            "What is the current weather, forecast, and active-alert "
            "status for Chicago, IL?"
        ),
        "expected_specialists": {"weather_agent"},
        "expected_tools": {"geocode_place", "get_weather"},
        "requires_grounding": False,
    },
    {
        "label": "search_only",
        "prompt": (
            "What are the latest official NASA updates about the "
            "Artemis II mission? Include relevant publication dates."
        ),
        "expected_specialists": {"google_search_agent"},
        "expected_tools": set(),
        "requires_grounding": True,
    },
    {
        "label": "combined_weather_and_search",
        "prompt": (
            "Give me the current weather and active-alert status for "
            "Miami, FL, then find the latest official NOAA Atlantic "
            "hurricane outlook and summarize both."
        ),
        "expected_specialists": {
            "weather_agent",
            "google_search_agent",
        },
        "expected_tools": {"geocode_place", "get_weather"},
        "requires_grounding": True,
    },
    {
        "label": "unrelated_request",
        "prompt": "Write a limerick about database indexes.",
        "expected_specialists": set(),
        "expected_tools": set(),
        "requires_grounding": False,
    },
]

live_results: list[dict[str, Any]] = []
for case in LIVE_CASES:
    result = await run_multi_agent_case(
        case["prompt"],
        label=case["label"],
    )
    specialist_authors = SPECIALIST_NAMES & set(
        result["event_authors"]
    )
    specialist_starts = SPECIALIST_NAMES & set(
        result["started_agents"]
    )
    observed_tools = {
        call["name"] for call in result["tool_calls"]
    }

    assert result["final_answer"], result
    assert specialist_authors == case["expected_specialists"], result
    assert specialist_starts == case["expected_specialists"], result
    assert case["expected_specialists"] <= set(
        result["transfer_targets"]
    ), result
    assert case["expected_tools"] <= observed_tools, result
    if case["requires_grounding"]:
        assert result["google_search_grounded"] is True, result
    if not case["expected_specialists"]:
        assert observed_tools == set(), result

    live_results.append(result)

assert len({item["session_id"] for item in live_results}) == len(
    live_results
)
print(json.dumps(live_results, indent=2))


/opt/micromamba/lib/python3.12/site-packages/google/adk/tools/transfer_to_agent_tool.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  function_decl = super()._get_declaration()
Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


[
  {
    "label": "weather_only",
    "prompt": "What is the current weather, forecast, and active-alert status for Chicago, IL?",
    "session_id": "task3-weather_only-4c660d0ebcfa",
    "started_agents": [
      "root_agent",
      "weather_agent"
    ],
    "event_authors": [
      "root_agent",
      "weather_agent"
    ],
    "transfer_targets": [
      "weather_agent"
    ],
    "tool_calls": [
      {
        "author": "weather_agent",
        "name": "geocode_place",
        "arguments": {
          "place": "Chicago, IL, USA"
        }
      },
      {
        "author": "weather_agent",
        "name": "get_weather",
        "arguments": {
          "latitude": 41.88325,
          "longitude": -87.632388
        }
      }
    ],
    "google_search_grounded": false,
    "final_answer": "For Chicago, IL:\nCurrent observation: Mostly Clear, 78.8\u00b0F, humidity 47.6%, wind 62.2 mph.\nForecast for Today: Sunny, with a high near 76\u00b0F. Northeast wind 5 to 10 mph.\nActive-aler

## 7. Failure and boundary behavior

An invalid location must still route through the weather specialist,
call geocoding, avoid the downstream NWS call, and return a clear
nonempty explanation.


In [11]:
invalid_location_result = await run_multi_agent_case(
    (
        "What is the current weather for "
        "This Place Should Not Exist 9z8y7x6w5v?"
    ),
    label="invalid_location",
)
invalid_tool_names = [
    call["name"] for call in invalid_location_result["tool_calls"]
]
assert "weather_agent" in invalid_location_result["event_authors"]
assert "weather_agent" in invalid_location_result["transfer_targets"]
assert invalid_tool_names == ["geocode_place"], invalid_location_result
assert invalid_location_result["final_answer"], invalid_location_result
print(json.dumps(invalid_location_result, indent=2))


{
  "label": "invalid_location",
  "prompt": "What is the current weather for This Place Should Not Exist 9z8y7x6w5v?",
  "session_id": "task3-invalid_location-4ba70b79573f",
  "started_agents": [
    "root_agent",
    "weather_agent"
  ],
  "event_authors": [
    "root_agent",
    "weather_agent"
  ],
  "transfer_targets": [
    "weather_agent"
  ],
  "tool_calls": [
    {
      "author": "weather_agent",
      "name": "geocode_place",
      "arguments": {
        "place": "This Place Should Not Exist 9z8y7x6w5v"
      }
    }
  ],
  "google_search_grounded": false,
  "final_answer": "I'm sorry, I couldn't find a place called \"This Place Should Not Exist 9z8y7x6w5v.\" Please check the spelling or provide a valid U.S. location.",
  "events": [
    {
      "author": "root_agent",
      "branch": null,
      "transfer_to_agent": null,
      "function_calls": [
        {
          "name": "transfer_to_agent",
          "arguments": {
            "agent_name": "weather_agent"
          }


## 8. Grading evidence

The final assertions map the workshop and repository acceptance
criteria to saved live output. A passing cell is the notebook's
machine-checkable self-grade.


In [12]:
results_by_label = {item["label"]: item for item in live_results}
weather_live = results_by_label["weather_only"]
search_live = results_by_label["search_only"]
combined_live = results_by_label["combined_weather_and_search"]
unrelated_live = results_by_label["unrelated_request"]

evidence = {
    "copied_from_task_2": (
        TASK3_SOURCE_NOTEBOOK == "02_callbacks.ipynb"
    ),
    "three_agents_created": {
        root_agent.name,
        weather_agent.name,
        google_search_agent.name,
    }
    == {"root_agent", "weather_agent", "google_search_agent"},
    "coordinating_root_agent": root_agent.name == "root_agent",
    "weather_agent_uses_custom_tools": set(
        hierarchy_evidence["weather_tool_names"]
    )
    == {"geocode_place", "get_weather"},
    "search_agent_uses_builtin_google_search": hierarchy_evidence[
        "search_uses_builtin_google_search"
    ],
    "search_tool_boundary_is_valid": hierarchy_evidence[
        "search_is_terminal_specialist"
    ],
    "specialists_registered_as_sub_agents": hierarchy_evidence[
        "sub_agent_names"
    ]
    == ["weather_agent", "google_search_agent"],
    "root_delegated_weather_request": (
        "weather_agent" in weather_live["transfer_targets"]
        and "weather_agent" in weather_live["event_authors"]
    ),
    "root_delegated_search_request": (
        "google_search_agent" in search_live["transfer_targets"]
        and "google_search_agent" in search_live["event_authors"]
    ),
    "weather_tools_ran_live": {
        call["name"] for call in weather_live["tool_calls"]
    }
    >= {"geocode_place", "get_weather"},
    "google_search_ran_live": search_live["google_search_grounded"],
    "combined_request_used_both_specialists": SPECIALIST_NAMES
    <= set(combined_live["event_authors"]),
    "combined_request_used_both_sources": bool(
        combined_live["google_search_grounded"]
        and {
            call["name"] for call in combined_live["tool_calls"]
        }
        >= {"geocode_place", "get_weather"}
    ),
    "events_prove_sub_agent_use": all(
        item["events"] for item in live_results
    ),
    "unrelated_request_handled_by_root": (
        not (SPECIALIST_NAMES & set(unrelated_live["event_authors"]))
        and unrelated_live["final_answer"]
    ),
    "failure_case_is_safe_and_bounded": (
        [call["name"] for call in invalid_location_result["tool_calls"]]
        == ["geocode_place"]
        and bool(invalid_location_result["final_answer"])
    ),
    "fresh_sessions_for_every_live_case": len(
        {
            item["session_id"]
            for item in [*live_results, invalid_location_result]
        }
    )
    == len(live_results) + 1,
    "all_live_responses_nonempty": all(
        item["final_answer"]
        for item in [*live_results, invalid_location_result]
    ),
}

assert all(evidence.values()), evidence
print(json.dumps(evidence, indent=2))
print("TASK 3 COMPLETE: all multi-agent grading checks passed.")


{
  "copied_from_task_2": true,
  "three_agents_created": true,
  "coordinating_root_agent": true,
  "weather_agent_uses_custom_tools": true,
  "search_agent_uses_builtin_google_search": true,
  "search_tool_boundary_is_valid": true,
  "specialists_registered_as_sub_agents": true,
  "root_delegated_weather_request": true,
  "root_delegated_search_request": true,
  "weather_tools_ran_live": true,
  "google_search_ran_live": true,
  "combined_request_used_both_specialists": true,
  "combined_request_used_both_sources": true,
  "events_prove_sub_agent_use": true,
  "unrelated_request_handled_by_root": "I cannot fulfill that request. My capabilities are limited to providing live U.S. weather information and performing current-information web searches.",
  "failure_case_is_safe_and_bounded": true,
  "fresh_sessions_for_every_live_case": true,
  "all_live_responses_nonempty": true
}
TASK 3 COMPLETE: all multi-agent grading checks passed.


## References

- [Google ADK multi-agent workflow patterns](https://adk.dev/workflows/patterns/)
- [Google ADK agent-team tutorial](https://adk.dev/tutorials/agent-team/)
- [Google Search tool for ADK](https://adk.dev/tools/gemini-api/google-search/)
- [Google ADK events](https://adk.dev/events/)
- [Google Maps Geocoding API](https://developers.google.com/maps/documentation/geocoding)
- [National Weather Service API](https://www.weather.gov/documentation/services-web-api)
